# General N-Direction Random Walk Solution

This notebook simulates a walk where each step picks one of N equally spaced directions:
angle_k = 2*pi*k/N, for k = 0..N-1.

Special case note: N = 4 is exactly the same model as the mathemagicland notebook.

Outputs are saved in `outputs/general_walk_solution/`.

In [1]:
from pathlib import Path
import json
import math
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration
RANDOM_SEED = 20260407
N_TRIALS = 300
MAX_STEPS = 20_000
N_DIRECTIONS = 6
RETURN_TOLERANCE = 1e-9
POSITION_ROUND_DECIMALS = 12

NOTEBOOK_NAME = 'general_walk_solution.ipynb'

cwd = Path.cwd()
if (cwd / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd
elif (cwd / 'random_walks' / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd / 'random_walks'
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'general_walk_solution'
PATHS_DIR = OUTPUT_DIR / 'paths'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATHS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Output directory:   {OUTPUT_DIR}')
print(f'N_DIRECTIONS:       {N_DIRECTIONS}')

Notebook directory: /home/manpazito/projects/fun_mini_sims/random_walks
Output directory:   /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution
N_DIRECTIONS:       6


## Mathematical Description and Return Detection

Position after n steps is the sum of selected unit vectors.

Floating-point issue:
- For some N values, exact coordinates are not perfectly representable, so checking x == 0 and y == 0 can be unreliable.

Robust approach used here:
- For N in {2, 4}, use exact axis vectors and exact equality checks.
- For other N values, use tolerance-based return detection with `RETURN_TOLERANCE` and controlled coordinate rounding.

This keeps return detection numerically stable while preserving the intended random-walk model.

In [2]:
def get_direction_vectors(n_directions: int) -> np.ndarray:
    # Return an (n_directions, 2) array of unit step vectors.
    if n_directions < 2:
        raise ValueError('N_DIRECTIONS must be >= 2')

    if n_directions == 2:
        return np.array([[1.0, 0.0], [-1.0, 0.0]], dtype=float)

    if n_directions == 4:
        return np.array(
            [
                [1.0, 0.0],
                [0.0, 1.0],
                [-1.0, 0.0],
                [0.0, -1.0],
            ],
            dtype=float,
        )

    angles = 2.0 * np.pi * np.arange(n_directions) / n_directions
    return np.column_stack((np.cos(angles), np.sin(angles)))


def reached_origin(x: float, y: float, n_directions: int, tolerance: float) -> bool:
    if n_directions in (2, 4):
        return (x == 0.0) and (y == 0.0)
    return math.hypot(x, y) <= tolerance


def simulate_general_walk(
    rng: np.random.Generator,
    n_directions: int,
    max_steps: int,
    return_tolerance: float,
    round_decimals: int,
) -> Dict[str, object]:
    # Simulate one N-direction walk until first return or max_steps.
    vectors = get_direction_vectors(n_directions)

    x, y = 0.0, 0.0
    xs: List[float] = [x]
    ys: List[float] = [y]

    min_x = max_x = x
    min_y = max_y = y
    max_distance = 0.0
    max_manhattan = 0.0
    returned_to_origin = False

    for _ in range(max_steps):
        direction_idx = int(rng.integers(0, n_directions))
        dx, dy = vectors[direction_idx]

        x += float(dx)
        y += float(dy)

        if n_directions not in (2, 4):
            x = float(np.round(x, decimals=round_decimals))
            y = float(np.round(y, decimals=round_decimals))

        xs.append(x)
        ys.append(y)

        min_x = min(min_x, x)
        max_x = max(max_x, x)
        min_y = min(min_y, y)
        max_y = max(max_y, y)

        distance = math.hypot(x, y)
        max_distance = max(max_distance, distance)
        max_manhattan = max(max_manhattan, abs(x) + abs(y))

        if reached_origin(x, y, n_directions, return_tolerance):
            returned_to_origin = True
            x, y = 0.0, 0.0
            xs[-1], ys[-1] = x, y
            break

    steps_until_stop = len(xs) - 1
    return_time = float(steps_until_stop) if returned_to_origin else np.nan
    bbox_width = max_x - min_x
    bbox_height = max_y - min_y

    return {
        'xs': xs,
        'ys': ys,
        'steps_until_stop': steps_until_stop,
        'returned_to_origin': returned_to_origin,
        'censored': not returned_to_origin,
        'return_time': return_time,
        'max_distance': float(max_distance),
        'max_manhattan': float(max_manhattan),
        'bbox_width': float(bbox_width),
        'bbox_height': float(bbox_height),
        'bbox_area': float(bbox_width * bbox_height),
        'final_x': float(x),
        'final_y': float(y),
    }


def save_path_json(xs: List[float], ys: List[float], trial_index: int, paths_dir: Path) -> Path:
    payload = {
        'trial_index': int(trial_index),
        'x': [float(v) for v in xs],
        'y': [float(v) for v in ys],
    }
    out_path = paths_dir / f'trial_{trial_index:04d}.json'
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f)
    return out_path


def plot_single_walk(xs: List[float], ys: List[float], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.plot(xs, ys, color='tab:blue', linewidth=2, alpha=0.95)
    ax.scatter([0], [0], color='green', marker='*', s=180, label='origin')
    ax.scatter([xs[-1]], [ys[-1]], color='red', marker='X', s=120, label='stop point')
    ax.set_title(f'Single {N_DIRECTIONS}-Direction Walk')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_return_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    completed = summary_df.loc[~summary_df['censored'], 'return_time']
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(completed, bins=30, color='tab:blue', edgecolor='white')
    ax.set_title('Return Time Histogram (Completed Trials)')
    ax.set_xlabel('Steps until first return')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_max_distance_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(summary_df['max_distance'], bins=30, color='tab:orange', edgecolor='white')
    ax.set_title('Max Euclidean Distance Histogram')
    ax.set_xlabel('max Euclidean distance during trial')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_overlay_walks(all_paths: List[Tuple[List[float], List[float]]], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 7))
    for xs, ys in all_paths:
        ax.plot(xs, ys, color='tab:blue', alpha=0.10, linewidth=0.8)

    ax.scatter([0], [0], color='black', marker='*', s=160, label='origin')
    ax.set_title(f'Overlay of All {N_DIRECTIONS}-Direction Walks')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

## Single Example Walk

Simulate one trajectory and plot it in 2D.

In [3]:
single_walk = simulate_general_walk(
    rng=rng,
    n_directions=N_DIRECTIONS,
    max_steps=MAX_STEPS,
    return_tolerance=RETURN_TOLERANCE,
    round_decimals=POSITION_ROUND_DECIMALS,
)

single_walk_plot_path = OUTPUT_DIR / 'single_walk.png'
plot_single_walk(single_walk['xs'], single_walk['ys'], single_walk_plot_path)

print('Single walk summary:')
print(f"  steps_until_stop:   {single_walk['steps_until_stop']}")
print(f"  returned_to_origin: {single_walk['returned_to_origin']}")
print(f"  max_distance:       {single_walk['max_distance']:.3f}")
print(f'  saved plot:         {single_walk_plot_path}')

Single walk summary:
  steps_until_stop:   2
  returned_to_origin: True
  max_distance:       1.000
  saved plot:         /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution/single_walk.png


## Monte Carlo Simulation

Run many independent trials, save per-trial raw paths, and write summary statistics to CSV.

In [4]:
trial_records: List[Dict[str, object]] = []
all_paths: List[Tuple[List[float], List[float]]] = []

for trial_idx in range(N_TRIALS):
    walk = simulate_general_walk(
        rng=rng,
        n_directions=N_DIRECTIONS,
        max_steps=MAX_STEPS,
        return_tolerance=RETURN_TOLERANCE,
        round_decimals=POSITION_ROUND_DECIMALS,
    )

    all_paths.append((walk['xs'], walk['ys']))
    save_path_json(walk['xs'], walk['ys'], trial_idx, PATHS_DIR)

    trial_records.append(
        {
            'trial_index': trial_idx,
            'n_directions': N_DIRECTIONS,
            'steps_until_stop': walk['steps_until_stop'],
            'return_time': walk['return_time'],
            'returned_to_origin': walk['returned_to_origin'],
            'censored': walk['censored'],
            'max_distance': walk['max_distance'],
            'max_manhattan': walk['max_manhattan'],
            'bbox_width': walk['bbox_width'],
            'bbox_height': walk['bbox_height'],
            'bbox_area': walk['bbox_area'],
            'final_x': walk['final_x'],
            'final_y': walk['final_y'],
        }
    )

summary_df = pd.DataFrame(trial_records)
summary_csv_path = OUTPUT_DIR / 'summary.csv'
summary_df.to_csv(summary_csv_path, index=False)

completed = summary_df[~summary_df['censored']]
print(f'Trials run:                {N_TRIALS}')
print(f'Completed returns:         {len(completed)}')
print(f'Censored trials:           {int(summary_df["censored"].sum())}')
if len(completed):
    print(f"Mean return time:          {completed['return_time'].mean():.3f}")
print(f"Mean max distance:         {summary_df['max_distance'].mean():.3f}")
print(f'Saved summary CSV:         {summary_csv_path}')

summary_df.head()

Trials run:                300
Completed returns:         218
Censored trials:           82
Mean return time:          773.009
Mean max distance:         56.349
Saved summary CSV:         /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution/summary.csv


,trial_index,n_directions,steps_until_stop,return_time,returned_to_origin,censored,max_distance,max_manhattan,bbox_width,bbox_height,bbox_area,final_x,final_y
0,0,6,21,21.0,True,False,4.000000,5.464102,3.5,3.464102,12.124356,0.0,0.000000
1,1,6,20000,NaN,False,True,129.085243,153.073684,169.5,146.358293,24807.730704,114.0,-3.464102
2,2,6,20000,NaN,False,True,249.511523,325.152522,180.5,246.817240,44550.511834,8.0,206.114046
3,3,6,20000,NaN,False,True,103.460137,145.674337,142.5,90.932667,12957.905104,8.0,45.033321
4,4,6,20000,NaN,False,True,178.277873,221.910852,103.0,218.238402,22478.555381,4.5,-148.090344


## Visualization of Monte Carlo Results

Histograms for return times and max Euclidean distances.

In [5]:
return_hist_path = OUTPUT_DIR / 'return_time_histogram.png'
max_dist_hist_path = OUTPUT_DIR / 'max_distance_histogram.png'

plot_return_histogram(summary_df, return_hist_path)
plot_max_distance_histogram(summary_df, max_dist_hist_path)

print(f'Saved: {return_hist_path}')
print(f'Saved: {max_dist_hist_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution/return_time_histogram.png
Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution/max_distance_histogram.png


## Final Overlay Plot

Overlay all simulated trajectories on one set of axes.

In [6]:
overlay_path = OUTPUT_DIR / 'overlay_walks.png'
plot_overlay_walks(all_paths, overlay_path)
print(f'Saved: {overlay_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/general_walk_solution/overlay_walks.png
